# Activation functions

In [1]:
import numpy as np

z = np.array([-4, -2, 0, 2, 4])

sig = 1 / (1 + np.exp(-z))
print(sig)
print(sig * (1 - sig))
print(np.tanh(z))

[0.01798621 0.11920292 0.5        0.88079708 0.98201379]
[0.01766271 0.10499359 0.25       0.10499359 0.01766271]
[-0.9993293  -0.96402758  0.          0.96402758  0.9993293 ]


In [2]:
print(np.maximum(0, z))
print(np.where(z>0, z, 0.01 * z))
print(np.where(z>0, 1.0, 0.01))

[0 0 0 2 4]
[-0.04 -0.02  0.    2.    4.  ]
[0.01 0.01 0.01 1.   1.  ]


In [3]:
import torch
import torch.nn.functional as F

for f in (F.relu, F.leaky_relu):
    w = torch.tensor(1.0, requires_grad=True)
    h = f(w * 2 - 5)
    ((h -1 ) ** 2).backward()
    print(h.item(), w.grad.item())

0.0 0.0
-0.029999999329447746 -0.04119999706745148


In [5]:
# mse and mae
x = np.arange(1, 6)
y = np.array([2, 4, 6, 8, 40])

for a, b in ((2, 0), (8, -12)):
    p = a* x + b
    print(a, b, np.mean(y - p) ** 2, np.mean(np.abs(y - p)))

2 0 36.0 6.0
8 -12 0.0 7.2


In [8]:
def softmax(z):
    e = np.exp(z- np.max(z))
    return e / np.sum(e)

print(softmax(np.array([2., 1., 0.,])))
print(softmax(np.array([1000., 1001.])))

[0.66524096 0.24472847 0.09003057]
[0.26894142 0.73105858]


## Working with wine dataset

In [9]:
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
sc = StandardScaler().fit(X_train)
A, B = [torch.tensor(sc.transform(v)).float() for v in (X_train, X_test)]

ya, yb = torch.tensor(y_train), torch.tensor(y_test)

print(X.shape, tuple(A.shape), tuple(B.shape))

(178, 13) (142, 13) (36, 13)


In [21]:
ACTS = {
    'sigmoid': nn.Sigmoid,
    'tanh': nn.Tanh,
    'relu': nn.ReLU,
    'leaky_relu': nn.LeakyReLU,
}

def fit_eval(act, lr=0.05, seed=42, loss_fn=F.cross_entropy):
    torch.manual_seed(seed)

    a = ACTS[act]

    net = nn.Sequential(
        nn.Linear(13, 16),
        a(),
        nn.Linear(16, 16),
        a(),
        nn.Linear(16, 16),
        a(),
        nn.Linear(16, 3)
    )

    opt = torch.optim.SGD(net.parameters(), lr=lr)

    for _ in range(100):
        loss = loss_fn(net(A), ya)

        opt.zero_grad()
        loss.backward()
        opt.step()

    with torch.no_grad():
        accuracy = (net(B).argmax(1) == yb).float().mean().item()

    return accuracy

In [22]:
for lr in (0.05, 0.09):
    for act in ACTS:
        print(lr, act, fit_eval(act, lr))

0.05 sigmoid 0.3888888955116272
0.05 tanh 1.0
0.05 relu 0.9444444179534912
0.05 leaky_relu 0.9722222089767456
0.09 sigmoid 0.3888888955116272
0.09 tanh 0.9722222089767456
0.09 relu 0.9722222089767456
0.09 leaky_relu 0.9722222089767456
